In [2]:
import pandas as pd
import numpy as np
import os
from tqdm import tqdm
import math
import re
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.preprocessing import OneHotEncoder

import sys
# sys.path.append('../..')
# from mount_drive import mount_s_drive

In [3]:
# mount_s_drive(subfolder='LCICM/Databases')

In [4]:
myHours = 60*6

In [5]:
database_folder = '/projects/LCICM/'

### Patients 

In [6]:
ca_ed_df = pd.read_csv('CA_ED.csv')
ca_time_df = pd.read_csv('CA_time.csv')

In [7]:
ids_df1 = ca_ed_df[['subject_id','hadm_id']]
ids_df2 = ca_time_df[['subject_id','hadm_id']]
myIds = pd.concat([ids_df1, ids_df2], ignore_index=True)
myIds = myIds.drop_duplicates()

### myPredictorsDf

In [8]:
ca_ed_df['time'] = ca_ed_df['intime']
ca_time_df['time'] = ca_time_df['max_time']

In [9]:
ca_df = pd.concat([ca_ed_df[['subject_id', 'hadm_id', 'gender', 'anchor_age', 'long_title', 'hospital_expire_flag', 'time']],
                   ca_time_df[['subject_id', 'hadm_id', 'gender', 'anchor_age', 'long_title', 'hospital_expire_flag', 'time']]])
ca_df = ca_df.sort_values(['subject_id','hadm_id'])
ca_df = ca_df.drop_duplicates(['subject_id','hadm_id']).reset_index(drop=True)
duplicate_subjects = ca_df.groupby('subject_id')['hadm_id'].nunique()
duplicate_subjects = duplicate_subjects[duplicate_subjects > 1].index
ca_df = ca_df[~ca_df['subject_id'].isin(duplicate_subjects)]

In [10]:
ca_df

,subject_id,hadm_id,gender,anchor_age,long_title,hospital_expire_flag,time
0,10004720,22081550.0,M,61,NaN,1,2186-11-12 19:55:00
1,10010471,29842315.0,F,89,Cardiac arrest due to underlying cardiac condi...,1,2155-12-02 20:33:00
2,10038688,25926997.0,M,46,NaN,1,2181-12-06 08:14:00
3,10056037,21758160.0,F,68,NaN,1,2131-03-31 03:20:00
4,10067389,23577021.0,M,76,Cardiac arrest due to other underlying condition,1,2116-02-10 23:55:00
...,...,...,...,...,...,...,...
1002,19926342,23717261.0,M,87,NaN,1,2166-04-12 22:07:00
1003,19948220,23370065.0,M,75,"Cardiac arrest, cause unspecified",1,2163-05-14 08:20:53
1004,19960823,25504764.0,F,60,NaN,0,NaN
1005,19962126,21472938.0,M,66,NaN,0,2145-02-20 14:27:00


In [138]:
myPredictorsDf = ca_df[['subject_id', 'gender', 'anchor_age', 'long_title', 'hospital_expire_flag']]
myPredictorsDf['gender'] = (myPredictorsDf['gender'] == 'M').astype(int)
myPredictorsDf.rename(columns={'anchor_age': 'age', 'long_title': 'diagnosis', 'hospital_expire_flag': 'death_at_disch'}, inplace=True)
myPredictorsDf = myPredictorsDf.reset_index(drop=True)

/local/mbranda1/3085937/ipykernel_2780644/1200390782.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  myPredictorsDf['gender'] = (myPredictorsDf['gender'] == 'M').astype(int)
/local/mbranda1/3085937/ipykernel_2780644/1200390782.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  myPredictorsDf.rename(columns={'anchor_age': 'age', 'long_title': 'diagnosis', 'hospital_expire_flag': 'death_at_disch'}, inplace=True)


In [139]:
myPredictorsDf.head()

,subject_id,gender,age,diagnosis,death_at_disch
0,10004720,1,61,NaN,1
1,10010471,0,89,Cardiac arrest due to underlying cardiac condi...,1
2,10038688,1,46,NaN,1
3,10056037,0,68,NaN,1
4,10067389,1,76,Cardiac arrest due to other underlying condition,1


### Extraction Functions

In [13]:
def getFeaturesFromDf(aDf, aTimeColumn, aTypeColumn, aValueColumn):
    aDf['num_values'] = pd.to_numeric(aDf[aValueColumn], errors='coerce')
    myMasterGroupDf = aDf[(aDf[aTimeColumn] < myHours)].groupby(['subject_id', aTypeColumn])
    myMasterGroupBegin = aDf.loc[myMasterGroupDf[aTimeColumn].idxmin()]
    myMasterGroupEnd = aDf.loc[myMasterGroupDf[aTimeColumn].idxmax()]
    myMasterGroupAgg = myMasterGroupDf.agg({'mean', 'min', 'max'})
    myMasterGroupAgg = myMasterGroupAgg.num_values.reset_index()
    return myMasterGroupDf, myMasterGroupBegin, myMasterGroupEnd, myMasterGroupAgg

In [14]:
def mergeFeaturesInDf(aDfToMerge, aBegin, aEnd, aAgg, aPrefix, aTypeColumn, aValueColumn):
    subject_ids = aDfToMerge[['subject_id']].copy()
    myPredictorsDf1 = aDfToMerge.copy()
    
    print('Running Begin Columns')
    total = int(aBegin[aTypeColumn].nunique() / 10)
    begin_cols = {}
    i = 0
    print('progress: ', end="")
    for value in aBegin[aTypeColumn].unique():
        myDf = aBegin[aBegin[aTypeColumn] == value]
        merged = subject_ids.merge(myDf[['subject_id', aValueColumn]], on='subject_id', how='left')
        begin_cols[aPrefix + '_first_' + value] = merged[aValueColumn]
        if (i % total == 0):
            print('#', end="")
        i+= 1
    begin_df = pd.concat([subject_ids, pd.DataFrame(begin_cols)], axis=1)
    print()

    print('Running End Columns')
    total = int(aEnd[aTypeColumn].nunique() / 10)
    end_cols = {}
    i = 0
    print('progress: ', end="")
    for value in aEnd[aTypeColumn].unique():
        myDf = aEnd[aEnd[aTypeColumn] == value]
        merged = subject_ids.merge(myDf[['subject_id', aValueColumn]], on='subject_id', how='left')
        end_cols[aPrefix + '_last_' + value] = merged[aValueColumn]
        if (i % total == 0):
            print('#', end="")
        i+= 1
    end_df = pd.concat([subject_ids, pd.DataFrame(end_cols)], axis=1)
    print()
    
    print('Running Agg Columns')
    total = int(aAgg[aTypeColumn].nunique() / 10)
    agg_cols = {}
    i = 0
    print('progress: ', end="")
    for value in aAgg[aTypeColumn].unique():
        myDf = aAgg[aAgg[aTypeColumn] == value]
        merged = subject_ids.merge(myDf[['subject_id', 'max', 'min', 'mean']], on='subject_id', how='left')
        agg_cols.update({aPrefix + '_max_' + value: merged['max'],
                         aPrefix + '_min_' + value: merged['min'],
                         aPrefix + '_mean_' + value: merged['mean']})
        if (i % total == 0):
            print('#', end="")
        i+= 1
    agg_df = pd.concat([subject_ids, pd.DataFrame(agg_cols)], axis=1)
    print()

    myPredictorsDf1 = myPredictorsDf1.merge(begin_df, on='subject_id', how='left')
    myPredictorsDf1 = myPredictorsDf1.merge(end_df, on='subject_id', how='left')
    myPredictorsDf1 = myPredictorsDf1.merge(agg_df, on='subject_id', how='left')
    
    return myPredictorsDf1

In [15]:
def read_by_chunks(file_path, myIds, chunk_size=1e6):
    df_chunks = []
    num_chunks = 0    
    for chunk in pd.read_csv(file_path,chunksize=chunk_size):
        chunk = chunk[chunk['subject_id'].isin(myIds['subject_id'])]
        df_chunks.append(chunk)
        if num_chunks % 20 == 0:
            print(f'Chunk {num_chunks} Processed.')
        num_chunks += 1
        del chunk
    print('Processing finished.')    
    return pd.concat(df_chunks, ignore_index=True)

### Chartevents

In [16]:
d_items_df = pd.read_csv(database_folder+'mimic-iv-2.2/icu/d_items.csv')
d_items_df['label'] = d_items_df['label'].str.lower().str.replace(' ', '_')

In [17]:
chartevents_df = read_by_chunks(database_folder+'mimic-iv-2.2/icu/chartevents.csv', myIds)

Chunk 0 Processed.
Chunk 20 Processed.
Chunk 40 Processed.
Chunk 60 Processed.
Chunk 80 Processed.
Chunk 100 Processed.
Chunk 120 Processed.
Chunk 140 Processed.
Chunk 160 Processed.
Chunk 180 Processed.
Chunk 200 Processed.
Chunk 220 Processed.
Chunk 240 Processed.
Chunk 260 Processed.
Chunk 280 Processed.
Chunk 300 Processed.
Processing finished.


In [18]:
chartevents_df = pd.merge(chartevents_df,ca_df[['subject_id','time']],on='subject_id',how='right')
chartevents_df['charttime'] = pd.to_datetime(chartevents_df['charttime'],errors='coerce')
chartevents_df['time'] = pd.to_datetime(chartevents_df['time'],errors='coerce')
chartevents_df['chartoffset'] = (chartevents_df['charttime']-chartevents_df['time']).dt.total_seconds()/60
chartevents_df = chartevents_df[chartevents_df['chartoffset']>=0]
chartevents_df = chartevents_df.sort_values(by=['subject_id','chartoffset'])
chartevents_df = chartevents_df.merge(d_items_df[['itemid','label','param_type']], on='itemid', how='left')

In [19]:
columns_to_keep = ['subject_id','value','chartoffset','itemid','label','param_type']
chart_df = chartevents_df[columns_to_keep]
chart_df = chart_df.dropna(subset='label')
chart_df = chart_df[chart_df['chartoffset']<=myHours]

In [20]:
num_df = chart_df[chart_df['param_type']=='Numeric']
labs_df = chart_df[chart_df['param_type']=='Numeric with tag']
bin_df = chart_df[chart_df['param_type']=='Checkbox']
cat_df = chart_df[chart_df['param_type']=='Text']

In [21]:
num_df = num_df.drop(columns=['param_type'])
labs_df = labs_df.drop(columns=['param_type'])
bin_df = bin_df.drop(columns=['param_type'])
cat_df = cat_df.drop(columns=['param_type'])

In [100]:
bin_df

,chart_14_gauge_dressing_occlusive,chart_14_gauge_placed_in_outside_facility,chart_14_gauge_placed_in_the_field,chart_16_gauge_dressing_occlusive,chart_16_gauge_placed_in_outside_facility,chart_16_gauge_placed_in_the_field,chart_18_gauge_dressing_occlusive,chart_18_gauge_placed_in_outside_facility,chart_18_gauge_placed_in_the_field,chart_20_gauge_dressing_occlusive,...,chart_unable_to_assess_nutrition_/_education,chart_unable_to_assess_pain,chart_unable_to_assess_psychological,chart_unable_to_assess_teaching_/_learning_needs,chart_unable_to_obtain,chart_unintentional_weight_loss_>10_lbs.,chart_vaulables_checklist,chart_vented,chart_vibration_in_line,chart_visual_/_hearing_deficit
subject_id,,,,,,,,,,,,,,,,,,,,,
10004720,NaN,NaN,NaN,NaN,NaN,NaN,1.0,0.0,0.0,NaN,...,NaN,NaN,NaN,1.0,NaN,0.0,NaN,NaN,NaN,0.0
10010471,NaN,NaN,NaN,NaN,NaN,NaN,1.0,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10038688,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,1.0,NaN,...,1.0,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10067389,NaN,NaN,NaN,1.0,1.0,0.0,NaN,NaN,NaN,NaN,...,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN
10079545,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,0.0,NaN,...,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19904446,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,...,NaN,1.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN
19914761,NaN,NaN,NaN,1.0,0.0,0.0,NaN,NaN,NaN,NaN,...,1.0,NaN,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN
19926342,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [105]:
[x for x in bin_df.columns if 'ven' in x.lower()]

['chart_high_risk_(>51)_interventions',
 'chart_intravenous__/_iv_access_prior_to_admission',
 'chart_vented']

In [95]:
cat_df[d_items_df.abbreviation.str.lower().str.contains('fibri')]

,itemid,label,abbreviation,linksto,category,unitname,param_type,lownormalvalue,highnormalvalue
135,220541,zfibrinogen,ZFibrinogen,chartevents,Labs,NaN,Numeric,NaN,NaN
258,221200,fibrini,Fibrini,inputevents,Fluids - Other (Not In Use),mL,Solution,NaN,NaN
259,221201,fibrini_energy,Fibrini Energy,inputevents,Fluids - Other (Not In Use),mL,Solution,NaN,NaN
1318,225464,cardioversion/defibrillation,Cardioversion/Defibrillation,procedureevents,3-Significant Events,NaN,Processes,NaN,NaN
2229,227468,fibrinogen,Fibrinogen,chartevents,Labs,NaN,Numeric with tag,NaN,NaN


In [87]:
cat_df

,subject_id,value,chartoffset,itemid,label
3,10004720,Yes,5.0,223781.0,pain_present
4,10004720,Intermittent,5.0,223782.0,pain_type
5,10004720,Generalized,5.0,223783.0,pain_location
6,10004720,At Rest,5.0,223784.0,pain_cause
7,10004720,Unable to Score,5.0,223791.0,pain_level
...,...,...,...,...,...
7791579,19993336,Supine,328.0,224093.0,position
7791580,19993336,Oral,328.0,224642.0,temperature_site
7791581,19993336,2a - Passive or Active ROM,328.0,229321.0,activity_/_mobility_(jh-hlm)
7791588,19993336,V Paced,328.0,220048.0,heart_rhythm


#### Num

In [22]:
num_df['value'] = pd.to_numeric(num_df['value'], errors='coerce')
num_df = num_df.dropna(subset=['value'])
num_df['itemid'].nunique()

347

In [23]:
myMasterGroupDf, myMasterGroupBegin, myMasterGroupEnd, myMasterGroupAgg = getFeaturesFromDf(num_df, 'chartoffset', 'label', 'value')
myPredictorsDf1 = mergeFeaturesInDf(myPredictorsDf, myMasterGroupBegin, myMasterGroupEnd, myMasterGroupAgg, 'chart', 'label', 'value')

Running Begin Columns
progress: ###########
Running End Columns
progress: ###########
Running Agg Columns
progress: ###########


#### Bin

In [24]:
bin_df['value'] = pd.to_numeric(bin_df['value'], errors='coerce')
bin_df = bin_df.dropna(subset=['value'])
bin_df['value'] = bin_df['value'].astype(int)
bin_df['itemid'].nunique()

161

In [25]:
bin_df = bin_df.groupby(['subject_id','label']).max().reset_index()
bin_df = bin_df.pivot_table(index='subject_id', columns='label', values='value')
bin_df.columns = ['chart_' + col for col in bin_df.columns]

In [26]:
myPredictorsDf2 = myPredictorsDf.merge(bin_df, on='subject_id', how='left')
myPredictorsDf2 = myPredictorsDf2.fillna(0)

#### Cat

In [27]:
cat_df = cat_df.dropna(subset=['value'])
cat_df['value'] = cat_df['value'].str.split(';')
cat_df = cat_df.explode('value')
cat_df['itemid'].nunique()

945

In [28]:
cat_df_agg = cat_df.groupby(['subject_id', 'label'])['value'].agg(list).reset_index()
pivoted_df = cat_df_agg.pivot(index='subject_id', columns='label', values='value').fillna('')
encoded_dict = {}
for label in pivoted_df.columns:
    mlb = MultiLabelBinarizer()
    encoded = mlb.fit_transform(pivoted_df[label])
    encoded_dict.update({f"chart_{label}_{cls}": encoded[:, i] for i, cls in enumerate(mlb.classes_)})
binary_df = pd.DataFrame(encoded_dict, index=pivoted_df.index)
binary_df.reset_index(inplace=True)

In [29]:
myPredictorsDf3 = myPredictorsDf.merge(binary_df, on='subject_id', how='left')
myPredictorsDf3 = myPredictorsDf3.fillna(0)

#### Labs

In [30]:
labs_df['value'] = pd.to_numeric(labs_df['value'], errors='coerce')
labs_df = labs_df.dropna(subset=['value'])
labs_df['itemid'].nunique()

22

In [31]:
myMasterGroupDf, myMasterGroupBegin, myMasterGroupEnd, myMasterGroupAgg = getFeaturesFromDf(labs_df, 'chartoffset', 'label', 'value')
myPredictorsDf4 = mergeFeaturesInDf(myPredictorsDf, myMasterGroupBegin, myMasterGroupEnd, myMasterGroupAgg, 'lab', 'label', 'value')

Running Begin Columns
progress: ###########
Running End Columns
progress: ###########
Running Agg Columns
progress: ###########


### Outputevents

In [32]:
outputevents_df = read_by_chunks(database_folder+'mimic-iv-2.2/icu/outputevents.csv', myIds)

Chunk 0 Processed.
Processing finished.


In [33]:
outputevents_df = pd.merge(outputevents_df,ca_df[['subject_id','time']],on='subject_id',how='right')
outputevents_df['charttime'] = pd.to_datetime(outputevents_df['charttime'],errors='coerce')
outputevents_df['time'] = pd.to_datetime(outputevents_df['time'],errors='coerce')
outputevents_df['chartoffset'] = (outputevents_df['charttime']-outputevents_df['time']).dt.total_seconds()/60
outputevents_df = outputevents_df[outputevents_df['chartoffset']>=0]
outputevents_df = outputevents_df.sort_values(by=['subject_id','chartoffset'])
outputevents_df = outputevents_df.merge(d_items_df[['itemid','label','param_type']], on='itemid', how='left')

In [34]:
columns_to_keep = ['subject_id','value','chartoffset','itemid','label','param_type']
output_df = outputevents_df[columns_to_keep]
output_df = output_df.dropna(subset='label')
output_df = output_df[output_df['chartoffset']<=myHours]

In [35]:
num_output_df = output_df[output_df['param_type']=='Numeric']
cat_output_df = output_df[output_df['param_type']=='Text']

In [36]:
num_output_df = num_output_df.drop(columns=['param_type'])
cat_output_df = cat_output_df.drop(columns=['param_type'])

#### Num

In [37]:
num_output_df['value'] = pd.to_numeric(num_output_df['value'], errors='coerce')
num_output_df = num_output_df.dropna(subset=['value'])
num_output_df['itemid'].nunique()

38

In [38]:
myMasterGroupDf, myMasterGroupBegin, myMasterGroupEnd, myMasterGroupAgg = getFeaturesFromDf(num_df, 'chartoffset', 'label', 'value')
myPredictorsDf5 = mergeFeaturesInDf(myPredictorsDf, myMasterGroupBegin, myMasterGroupEnd, myMasterGroupAgg, 'output', 'label', 'value')

Running Begin Columns
progress: ###########
Running End Columns
progress: ###########
Running Agg Columns
progress: ###########


#### Cat

In [39]:
cat_output_df['itemid'].nunique()

0

### Inputevents

In [40]:
inputevents_df = read_by_chunks(database_folder+'mimic-iv-2.2/icu/inputevents.csv', myIds)

Chunk 0 Processed.
Processing finished.


In [41]:
inputevents_df = pd.merge(inputevents_df,ca_df[['subject_id','time']],on='subject_id',how='right')
inputevents_df['starttime'] = pd.to_datetime(inputevents_df['starttime'],errors='coerce')
inputevents_df['time'] = pd.to_datetime(inputevents_df['time'],errors='coerce')
inputevents_df['offset'] = (inputevents_df['starttime']-inputevents_df['time']).dt.total_seconds()/60
inputevents_df = inputevents_df[inputevents_df['offset']>=0]
inputevents_df = inputevents_df.sort_values(by=['subject_id','offset'])
inputevents_df = inputevents_df.merge(d_items_df[['itemid','label','param_type']], on='itemid', how='left')

In [42]:
columns_to_keep = ['subject_id','offset','itemid','label']
input_df = inputevents_df[columns_to_keep]
input_df = input_df.dropna(subset='label')
input_df = input_df[input_df['offset']<=myHours]

In [43]:
encoder = OneHotEncoder(sparse_output=False, dtype=int)
encoded_matrix = encoder.fit_transform(input_df[['label']])
encoded_df = pd.DataFrame(encoded_matrix, columns=encoder.get_feature_names_out(['label']))
encoded_df.rename(columns=lambda col: col.replace('label_', 'input_'), inplace=True)
encoded_df['subject_id'] = input_df['subject_id'].values
bin_input_df = encoded_df.groupby('subject_id').max().reset_index()

In [44]:
myPredictorsDf6 = myPredictorsDf.merge(bin_input_df, on='subject_id', how='left')
myPredictorsDf6 = myPredictorsDf6.fillna(0)

### Medication Administrations

In [45]:
med_admin_df = read_by_chunks(database_folder+'mimic-iv-2.2/hosp/emar.csv', myIds)

Chunk 0 Processed.
Chunk 20 Processed.
Processing finished.


In [46]:
med_admin_df = med_admin_df.dropna(subset=['medication'])

In [47]:
med_admin_df = pd.merge(med_admin_df,ca_df[['subject_id','time']],on='subject_id',how='right')
med_admin_df['charttime'] = pd.to_datetime(med_admin_df['charttime'],errors='coerce')
med_admin_df['time'] = pd.to_datetime(med_admin_df['time'],errors='coerce')
med_admin_df['chartoffset'] = (med_admin_df['charttime']-med_admin_df['time']).dt.total_seconds()/60
med_admin_df = med_admin_df[med_admin_df['chartoffset']>=0]
med_admin_df = med_admin_df.sort_values(by=['subject_id','chartoffset'])

In [48]:
columns_to_keep = ['subject_id','chartoffset','medication']
meds_df = med_admin_df[columns_to_keep]
meds_df = meds_df[meds_df['chartoffset']<=myHours]
meds_df['medication'] = meds_df['medication'].str.lower().str.replace(' ', '_')

In [49]:
encoder = OneHotEncoder(sparse_output=False, dtype=int)
encoded_matrix = encoder.fit_transform(meds_df[['medication']])
encoded_df = pd.DataFrame(encoded_matrix, columns=encoder.get_feature_names_out(['medication']))
encoded_df.rename(columns=lambda col: col.replace('medication_', 'med_'), inplace=True)
encoded_df['subject_id'] = meds_df['subject_id'].values
bin_meds_df = encoded_df.groupby('subject_id').max().reset_index()

In [50]:
myPredictorsDf7 = myPredictorsDf.merge(bin_meds_df, on='subject_id', how='left')
myPredictorsDf7 = myPredictorsDf7.fillna(0)

### Other Diagnoses

In [51]:
def nameSearchCardiacArrest(text):
    text = str(text).lower()
    patterns = [r'cardia.*rrest',
                r'cardio.*rrest',
                r'circulat.*rrest',
                r'.*asystole.*',
                r'.*asystolia.*',
                r'\bpea\b|pulseless elec.*act.*']
    if any(re.search(pattern, text) for pattern in patterns):
        exclusion_patterns = [r'history|hx|h/o',
                              r'before cardiac arrest',
                              r'due to cardiac arrest',
                              r'neonatal',
                              r'newborn']       
        if not any(re.search(pattern, text) for pattern in exclusion_patterns):
            return True   
    return False

def icdSearchCardiacArrest(text):
    text = str(text).lower()
    icd10_match = re.search(r'\bi46.*\b', text)
    icd9_match = re.search(r'\b4275\b', text)
    return bool(icd10_match or icd9_match)

In [52]:
import pandas as pd

In [106]:
dx_id_df = pd.read_csv(database_folder+'mimic-iv-2.2/hosp/d_icd_diagnoses.csv')
diagnoses_df = read_by_chunks(database_folder+'mimic-iv-2.2/hosp/diagnoses_icd.csv', myIds)

Chunk 0 Processed.
Processing finished.


In [120]:
dx_id_df['icd_code'] = dx_id_df['icd_code'].str.strip()
dx_id_df['long_title'] = dx_id_df['long_title'].str.strip()
dx_id_df['icd_search'] = dx_id_df['icd_code'].transform(icdSearchCardiacArrest)
dx_id_df['name_search'] = dx_id_df['long_title'].transform(nameSearchCardiacArrest)
filtered_dx_id_df = dx_id_df[(~dx_id_df['icd_search'])&(~dx_id_df['name_search'])]

In [121]:
diagnoses_df['icd_code'] = diagnoses_df['icd_code'].str.strip()
dx_df = diagnoses_df.merge(filtered_dx_id_df[['icd_code','long_title']], on='icd_code', how='inner')
dx_df['long_title'] = dx_df['long_title'].str.lower().str.replace(' ', '_', regex=True)

In [122]:
[x for x in dx_df['long_title'] if 'pulse' in x]

['encounter_for_checking_and_testing_of_cardiac_pacemaker_pulse_generator_[battery]']

In [123]:
dx_df

,subject_id,hadm_id,seq_num,icd_code,icd_version,long_title
0,10004720,22081550,1,J690,10,pneumonitis_due_to_inhalation_of_food_and_vomit
1,10004720,22081550,2,G931,10,"anoxic_brain_damage,_not_elsewhere_classified"
2,10004720,22081550,3,I82621,10,acute_embolism_and_thrombosis_of_deep_veins_of...
3,10004720,22081550,4,Z8674,10,personal_history_of_sudden_cardiac_arrest
4,10004720,22081550,5,J95811,10,postprocedural_pneumothorax
...,...,...,...,...,...,...
64659,19993336,24615303,8,I952,10,hypotension_due_to_drugs
64660,19993336,24615303,9,T447X5A,10,adverse_effect_of_beta-adrenoreceptor_antagoni...
64661,19993336,24615303,10,L259,10,"unspecified_contact_dermatitis,_unspecified_cause"
64662,19993336,24615303,11,M1990,10,"unspecified_osteoarthritis,_unspecified_site"


In [124]:
dx_df[dx_df['long_title'].str.contains('asys', case=False, na=False)]

,subject_id,hadm_id,seq_num,icd_code,icd_version,long_title


In [125]:
# encoder = OneHotEncoder(dtype=int)
# encoded_matrix = encoder.fit_transform(dx_df[['long_title']])
# encoded_df = pd.DataFrame(encoded_matrix, columns=encoder.get_feature_names_out(['long_title']))
# encoded_df.rename(columns=lambda col: col.replace('long_title_', 'dx_'), inplace=True)
# 
encoder = OneHotEncoder(dtype=int)
encoded_matrix = encoder.fit_transform(dx_df[['long_title']]).toarray()

encoded_df = pd.DataFrame(
    encoded_matrix,
    columns=encoder.get_feature_names_out(['long_title']),
    index=dx_df.index
)
encoded_df['subject_id'] = dx_df['subject_id'].values
bin_dx_df = encoded_df.groupby('subject_id').max().reset_index()

In [126]:
bin_dx_df.columns

Index(['subject_id', 'long_title_(idiopathic)_normal_pressure_hydrocephalus',
       'long_title_39_weeks_gestation_of_pregnancy',
       'long_title_abdominal_aneurysm,_ruptured',
       'long_title_abdominal_aneurysm_without_mention_of_rupture',
       'long_title_abdominal_aortic_aneurysm,_without_rupture',
       'long_title_abdominal_distension_(gaseous)',
       'long_title_abdominal_or_pelvic_swelling,_mass,_or_lump,_other_specified_site',
       'long_title_abdominal_pain,_epigastric',
       'long_title_abdominal_pain,_generalized',
       ...
       'long_title_wegener's_granulomatosis',
       'long_title_wheelchair_dependence', 'long_title_wheezing',
       'long_title_wild-type_transthyretin-related_(attr)_amyloidosis',
       'long_title_zoster_ocular_disease,_unspecified',
       'long_title_zoster_with_other_complications',
       'long_title_zoster_without_complications',
       'long_title_zygomatic_fracture,_left_side,_initial_encounter_for_closed_fracture',
       '

In [127]:
myPredictorsDf8 = myPredictorsDf.merge(bin_dx_df,on='subject_id',how='left')
myPredictorsDf8 = myPredictorsDf8.fillna(0)

In [61]:
# OMR
omr = read_by_chunks(database_folder+'mimic-iv-2.2/hosp/omr.csv', myIds=myIds)

Chunk 0 Processed.
Processing finished.


In [64]:
ca2

,subject_id,time,time_date
0,10004720,2186-11-12 19:55:00,2186-11-12
1,10010471,2155-12-02 20:33:00,2155-12-02
2,10038688,2181-12-06 08:14:00,2181-12-06
3,10056037,2131-03-31 03:20:00,2131-03-31
4,10067389,2116-02-10 23:55:00,2116-02-10
...,...,...,...
1002,19926342,2166-04-12 22:07:00,2166-04-12
1003,19948220,2163-05-14 08:20:53,2163-05-14
1004,19960823,NaT,NaT
1005,19962126,2145-02-20 14:27:00,2145-02-20


In [69]:
import pandas as pd

omr2 = omr.copy()
ca2 = ca_df[['subject_id', 'time']].copy()

omr2['chartdate'] = pd.to_datetime(omr2['chartdate'], errors='coerce')
ca2['time'] = pd.to_datetime(ca2['time'], errors='coerce')
ca2['time_date'] = ca2['time'].dt.normalize()

cols_keep = ['Height (Inches)', 'Weight (Lbs)', 'BMI (kg/m2)']
omr2 = omr2.loc[omr2['result_name'].isin(cols_keep)].copy()
omr2['result_value'] = pd.to_numeric(omr2['result_value'], errors='coerce')

ca2 = ca2.dropna(subset=['subject_id', 'time_date']).copy()
omr2 = omr2.dropna(subset=['subject_id', 'chartdate']).copy()

ca2 = ca2.reset_index(drop=True)
ca2['event_id'] = ca2.index

# merge by subject
tmp = ca2.merge(omr2, on='subject_id', how='left')

# keep only OMR rows that happened before or on the event date
tmp = tmp.loc[tmp['chartdate'] <= tmp['time_date']].copy()

# pick the latest valid OMR date before the event
last_dates = (
    tmp.groupby('event_id', as_index=False)['chartdate']
       .max()
       .rename(columns={'chartdate': 'matched_chartdate'})
)

# keep all OMR rows from that matched date
tmp = tmp.merge(last_dates, on='event_id', how='inner')
tmp = tmp.loc[tmp['chartdate'] == tmp['matched_chartdate']].copy()

# if same variable repeated on same day, keep first seq_num
tmp = tmp.sort_values(['event_id', 'result_name', 'seq_num'])
tmp = tmp.drop_duplicates(subset=['event_id', 'result_name'], keep='first')

# pivot wide
omr_wide = (
    tmp.pivot(index='event_id', columns='result_name', values='result_value')
       .reset_index()
       .rename(columns={
           'Height (Inches)': 'height_in',
           'Weight (Lbs)': 'weight_lbs',
           'BMI (kg/m2)': 'bmi'
       })
)

# merge back
result = ca2.merge(last_dates, on='event_id', how='left')
result = result.merge(omr_wide, on='event_id', how='left')

result['height_cm'] = result['height_in'] * 2.54
result['weight_kg'] = result['weight_lbs'] * 0.45359237

result.head()

,subject_id,time,time_date,event_id,matched_chartdate,bmi,height_in,weight_lbs,height_cm,weight_kg
0,10004720,2186-11-12 19:55:00,2186-11-12,0,2186-11-10,21.1,NaN,147.00,NaN,66.678078
1,10010471,2155-12-02 20:33:00,2155-12-02,1,2155-05-08,26.0,66.0,161.16,167.64,73.100946
2,10038688,2181-12-06 08:14:00,2181-12-06,2,NaT,NaN,NaN,NaN,NaN,NaN
3,10056037,2131-03-31 03:20:00,2131-03-31,3,NaT,NaN,NaN,NaN,NaN,NaN
4,10067389,2116-02-10 23:55:00,2116-02-10,4,NaT,NaN,NaN,NaN,NaN,NaN


In [71]:
myPredictorsDf9 = result

In [76]:
result.columns

Index(['subject_id', 'time', 'time_date', 'event_id', 'matched_chartdate',
       'bmi', 'height_in', 'weight_lbs', 'height_cm', 'weight_kg'],
      dtype='object')

### Merge

In [140]:
columns = ['subject_id', 'gender', 'age', 'diagnosis', 'death_at_disch']
myPredictorsDf = myPredictorsDf1
myPredictorsDf = myPredictorsDf.merge(myPredictorsDf2, on=columns, how='left')
myPredictorsDf = myPredictorsDf.merge(myPredictorsDf3, on=columns, how='left')
myPredictorsDf = myPredictorsDf.merge(myPredictorsDf4, on=columns, how='left')
myPredictorsDf = myPredictorsDf.merge(myPredictorsDf5, on=columns, how='left')
myPredictorsDf = myPredictorsDf.merge(myPredictorsDf6, on=columns, how='left')
myPredictorsDf = myPredictorsDf.merge(myPredictorsDf7, on=columns, how='left')
myPredictorsDf = myPredictorsDf.merge(myPredictorsDf8, on=columns, how='left')
myPredictorsDf = myPredictorsDf.merge(myPredictorsDf9, on='subject_id', how='left')

In [141]:
myPredictorsDf8

,subject_id,gender,age,diagnosis,death_at_disch,chart_first_absolute_count_-_basos,chart_first_absolute_count_-_eos,chart_first_absolute_count_-_lymphs,chart_first_absolute_count_-_monos,chart_first_absolute_count_-_neuts,...,long_title_wegener's_granulomatosis,long_title_wheelchair_dependence,long_title_wheezing,long_title_wild-type_transthyretin-related_(attr)_amyloidosis,"long_title_zoster_ocular_disease,_unspecified",long_title_zoster_with_other_complications,long_title_zoster_without_complications,"long_title_zygomatic_fracture,_left_side,_initial_encounter_for_closed_fracture","long_title_zygomatic_fracture,_unspecified_side,_initial_encounter_for_closed_fracture",long_title_zygomycosis_[phycomycosis_or_mucormycosis]
0,10004720,1,61,0,1,0.03,0.0,0.58,1.29,13.53,...,0,0,0,0,0,0,0,0,0,0
1,10010471,0,89,Cardiac arrest due to underlying cardiac condi...,1,0.00,0.0,0.00,0.00,0.00,...,0,0,0,0,0,0,0,0,0,0
2,10038688,1,46,0,1,0.00,0.0,0.00,0.00,0.00,...,0,1,0,0,0,0,0,0,0,0
3,10056037,0,68,0,1,0.00,0.0,0.00,0.00,0.00,...,0,0,0,0,0,0,0,0,0,0
4,10067389,1,76,Cardiac arrest due to other underlying condition,1,0.00,0.0,0.00,0.00,0.00,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
996,19926342,1,87,0,1,0.00,0.0,0.00,0.00,0.00,...,0,0,0,0,0,0,0,0,0,0
997,19948220,1,75,"Cardiac arrest, cause unspecified",1,0.00,0.0,0.00,0.00,0.00,...,0,0,0,0,0,0,0,0,0,0
998,19960823,0,60,0,0,0.00,0.0,0.00,0.00,0.00,...,0,0,0,0,0,0,0,0,0,0
999,19962126,1,66,0,0,0.00,0.0,0.42,0.28,13.21,...,0,0,0,0,0,0,0,0,0,0


### CA Diagnosis

In [142]:
myPredictorsDf['other_underlying_condition'] = (myPredictorsDf['diagnosis'] == 'Cardiac arrest due to other underlying condition').astype(int)
myPredictorsDf['underlying_cardiac_condition'] = (myPredictorsDf['diagnosis'] == 'Cardiac arrest due to underlying cardiac condition').astype(int)
myPredictorsDf['following_other_sur'] = (myPredictorsDf['diagnosis'] == 'Postprocedural cardiac arrest following other surgery').astype(int)
myPredictorsDf['following_cardiac_sur'] = (myPredictorsDf['diagnosis'] == 'Postprocedural cardiac arrest following cardiac surgery').astype(int)
myPredictorsDf = myPredictorsDf.drop(columns=['diagnosis'])

### GCS

In [143]:
mgcs_df = chartevents_df[chartevents_df['itemid']==223901]
mgcs_df['valuenum'] = pd.to_numeric(mgcs_df['valuenum'], errors='coerce')
mgcs_df = mgcs_df.dropna(subset=['valuenum'])
mgcs_df['subject_id'].nunique()

/local/mbranda1/3085937/ipykernel_2780644/3198510107.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  mgcs_df['valuenum'] = pd.to_numeric(mgcs_df['valuenum'], errors='coerce')


743

In [144]:
myGcsDf = mgcs_df.sort_values(by=['subject_id','chartoffset'])
myGcsDf['valuenum'] = myGcsDf.groupby('subject_id')['valuenum'].ffill().bfill()
myGcsDf['chartoffset2'] = myGcsDf.chartoffset
myGcsDf2 = myGcsDf.groupby('subject_id').agg({'chartoffset':'min', 'chartoffset2': 'max'})
myGcsDfMin = myGcsDf.merge(myGcsDf2, on=['subject_id', 'chartoffset'], how='inner')
myGcsDfMax = myGcsDf.merge(myGcsDf2, on=['subject_id', 'chartoffset2'], how='inner')
myGcsDfMin['valuenum'] = myGcsDfMin['valuenum'].astype(int)
myGcsDfMax['valuenum'] = myGcsDfMax['valuenum'].astype(int)

In [145]:
myPredictorsDf = myPredictorsDf.merge(myGcsDfMin[['subject_id', 'chartoffset', 'valuenum']], on='subject_id')
myPredictorsDf.rename(columns={'chartoffset': 'first_mGCS_time', 'valuenum': 'first_mGCS'}, inplace=True)
myPredictorsDf = myPredictorsDf.merge(myGcsDfMax[['subject_id', 'chartoffset2', 'valuenum']], on='subject_id')
myPredictorsDf.rename(columns={'chartoffset2': 'last_mGCS_time', 'valuenum': 'last_mGCS'}, inplace=True)

### Hypothermia

In [146]:
cooling_status_df = chartevents_df[chartevents_df['itemid']==225052]
cooling_status_df = cooling_status_df[cooling_status_df['chartoffset']<=1440]
cooling_status_df['on'] = (cooling_status_df['value']=='On').astype(int)

In [147]:
cooling_status_df = cooling_status_df.sort_values(['subject_id', 'chartoffset'])
cooling_status_df['on_shift'] = cooling_status_df.groupby('subject_id')['on'].shift(1, fill_value=0)
cooling_status_df['change'] = (cooling_status_df['on'] != cooling_status_df['on_shift']).astype(int)
cooling_status_df['group'] = cooling_status_df.groupby('subject_id')['change'].cumsum()
on_periods = cooling_status_df[cooling_status_df['on'] == 1]
duration_df = on_periods.groupby(['subject_id', 'group']).agg(start_offset=('chartoffset', 'min'),
                                                              end_offset=('chartoffset', 'max')).reset_index()
duration_df['duration'] = duration_df['end_offset'] - duration_df['start_offset']
longest_df = duration_df.loc[duration_df.groupby('subject_id')['duration'].idxmax()]
longest_df = longest_df[['subject_id', 'start_offset', 'end_offset', 'duration']]
hyp_df = longest_df[longest_df['duration']>=720]

In [148]:
myPredictorsDf['hypothermia'] = myPredictorsDf['subject_id'].isin(hyp_df['subject_id']).astype(int)
myPredictorsDf['hypothermia'].sum()

np.int64(280)

In [149]:
myPredictorsDf

,subject_id,gender,age,death_at_disch,chart_first_absolute_count_-_basos_x,chart_first_absolute_count_-_eos_x,chart_first_absolute_count_-_lymphs_x,chart_first_absolute_count_-_monos_x,chart_first_absolute_count_-_neuts_x,chart_first_alkaline_phosphate_x,...,weight_kg,other_underlying_condition,underlying_cardiac_condition,following_other_sur,following_cardiac_sur,first_mGCS_time,first_mGCS,last_mGCS_time,last_mGCS,hypothermia
0,10004720,1,61,1,0.03,0.00,0.58,1.29,13.53,149.0,...,66.678078,0,0,0,0,5.0,1,6485.0,2,1
1,10010471,0,89,1,NaN,NaN,NaN,NaN,NaN,NaN,...,73.100946,0,1,0,0,17.0,6,6717.0,6,0
2,10038688,1,46,1,NaN,NaN,NaN,NaN,NaN,190.0,...,NaN,0,0,0,0,37.0,3,23026.0,1,1
3,10067389,1,76,1,NaN,NaN,NaN,NaN,NaN,307.0,...,NaN,1,0,0,0,82.0,1,18485.0,1,0
4,10079545,0,89,1,0.04,0.01,0.65,0.70,13.04,115.0,...,NaN,0,0,0,0,56.0,1,928.0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
738,19904446,0,53,1,NaN,NaN,NaN,NaN,NaN,78.0,...,NaN,0,0,0,0,50.0,1,2334.0,1,1
739,19914761,1,86,1,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,0,0,0,0,47.0,1,47.0,1,0
740,19926342,1,87,1,NaN,NaN,NaN,NaN,NaN,101.0,...,77.800163,0,0,0,0,353.0,4,24113.0,6,0
741,19962126,1,66,0,0.00,0.00,0.42,0.28,13.21,58.0,...,65.181224,0,0,0,0,33.0,5,75933.0,6,0


### Save

In [137]:
myPredictorsDf.to_csv('MIMIC_Predictors2.csv', index=False)

In [86]:
[x for x in myPredictorsDf.columns if 'rythm' in x.lower()]

[]